In [2]:
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd


REGIONAL_FRACTION = 0.01
EXPECTED_COLUMNS = [
    "location",
    "timestamp",
    "method",
    "anomaly_score",
    "threshold",
    "is_anomaly",
]

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SEED_DIR = (
    REPO_ROOT
    / "outputs"
    / "pvgis_mtgflow"
    / "downstream_dense"
    / "seed_15"
)
TEST_CSV = SEED_DIR / "anomaly_scores.csv"
TRAIN_CSV = SEED_DIR / "train_anomaly_scores.csv"

for path in (TEST_CSV, TRAIN_CSV):
    if not path.is_file():
        raise FileNotFoundError(path)

print("Caricamento CSV MTGFlow...")
scores = pd.read_csv(TEST_CSV, parse_dates=["timestamp"])
train_scores = pd.read_csv(TRAIN_CSV, parse_dates=["timestamp"])
print(f"Score test caricati: {len(scores):,}")
print(f"Score training caricati: {len(train_scores):,}")


def analyse(frame, name):
    if list(frame.columns) != EXPECTED_COLUMNS:
        raise ValueError(f"{name}: schema inatteso: {list(frame.columns)}")
    if not pd.api.types.is_datetime64_any_dtype(frame["timestamp"]):
        frame["timestamp"] = pd.to_datetime(frame["timestamp"])
    if not pd.api.types.is_bool_dtype(frame["is_anomaly"]):
        raise TypeError(
            f"{name}: is_anomaly non booleano: {frame['is_anomaly'].dtype}"
        )

    location = (
        frame.groupby("location", sort=True)
        .agg(
            n_windows=("is_anomaly", "size"),
            n_anomaly=("is_anomaly", "sum"),
            mean_score=("anomaly_score", "mean"),
            threshold=("threshold", "first"),
            threshold_variants=("threshold", "nunique"),
        )
    )
    location["anomaly_rate"] = location["n_anomaly"] / location["n_windows"]

    regional = (
        frame.groupby("timestamp", sort=True)
        .agg(
            n_anomaly=("is_anomaly", "sum"),
            n_scored=("is_anomaly", "size"),
        )
    )
    regional["anomaly_fraction"] = regional["n_anomaly"] / regional["n_scored"]
    regional["is_rare"] = regional["anomaly_fraction"] >= REGIONAL_FRACTION

    rare = regional.loc[regional["is_rare"]].reset_index()
    if rare.empty:
        n_intervals = 0
        longest_interval = 0
    else:
        rare["_interval"] = (
            rare["timestamp"].diff().ne(pd.Timedelta(hours=1)).cumsum()
        )
        interval_lengths = rare.groupby("_interval").size()
        n_intervals = int(len(interval_lengths))
        longest_interval = int(interval_lengths.max())

    score_values = frame["anomaly_score"].to_numpy(dtype=np.float64, copy=False)
    threshold_values = frame["threshold"].to_numpy(dtype=np.float64, copy=False)
    flags = frame["is_anomaly"].to_numpy(dtype=bool, copy=False)
    score_quantiles = frame["anomaly_score"].quantile(
        [0, 0.01, 0.25, 0.5, 0.75, 0.99, 1]
    )
    rate_quantiles = location["anomaly_rate"].quantile(
        [0, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 1]
    )

    report = {
        "name": name,
        "rows": int(len(frame)),
        "n_locations": int(frame["location"].nunique()),
        "n_timestamps": int(len(regional)),
        "timestamp_start": regional.index.min().isoformat(),
        "timestamp_end": regional.index.max().isoformat(),
        "years": sorted(int(year) for year in frame["timestamp"].dt.year.unique()),
        "methods": sorted(frame["method"].astype(str).unique().tolist()),
        "null_counts": {
            column: int(frame[column].isna().sum()) for column in frame.columns
        },
        "finite_scores": bool(np.isfinite(score_values).all()),
        "finite_thresholds": bool(np.isfinite(threshold_values).all()),
        "decision_mismatches": int(
            np.count_nonzero(flags != (score_values > threshold_values))
        ),
        "node_anomalies": int(flags.sum()),
        "node_anomaly_rate": float(flags.mean()),
        "windows_per_location_min": int(location["n_windows"].min()),
        "windows_per_location_max": int(location["n_windows"].max()),
        "nodes_per_timestamp_min": int(regional["n_scored"].min()),
        "nodes_per_timestamp_max": int(regional["n_scored"].max()),
        "threshold_variants_per_location_max": int(
            location["threshold_variants"].max()
        ),
        "required_nodes_for_rare_event": int(
            math.ceil(regional["n_scored"].max() * REGIONAL_FRACTION)
        ),
        "regional_rare_timestamps": int(regional["is_rare"].sum()),
        "regional_rare_rate": float(regional["is_rare"].mean()),
        "rare_intervals": n_intervals,
        "longest_rare_interval_hours": longest_interval,
        "max_simultaneous_anomalous_nodes": int(regional["n_anomaly"].max()),
        "max_regional_anomaly_fraction": float(
            regional["anomaly_fraction"].max()
        ),
        "score_quantiles": {
            str(q): float(value) for q, value in score_quantiles.items()
        },
        "location_anomaly_rate_quantiles": {
            str(q): float(value) for q, value in rate_quantiles.items()
        },
    }
    return report, location, regional


test_report, test_locations, regional_test = analyse(scores, "test_2019")
train_report, train_locations, regional_train = analyse(
    train_scores, "train_2016_2018"
)

common = test_locations.index.intersection(train_locations.index)
threshold_difference = np.abs(
    test_locations.loc[common, "threshold"].to_numpy()
    - train_locations.loc[common, "threshold"].to_numpy()
)
cross_split = {
    "same_locations": bool(
        set(test_locations.index.astype(str))
        == set(train_locations.index.astype(str))
    ),
    "common_locations": int(len(common)),
    "threshold_mismatch_locations": int(
        np.count_nonzero(threshold_difference > 1e-12)
    ),
    "max_threshold_difference": float(threshold_difference.max()),
}

print("\n=== AUDIT COMPLETO ===")
print(
    json.dumps(
        {"test": test_report, "train": train_report, "cross_split": cross_split},
        indent=2,
    )
)
print("\n=== 20 TIMESTAMP TEST PIÙ ANOMALI ===")
print(regional_test.nlargest(20, "n_anomaly").reset_index().to_string(index=False))
print("\n=== 20 TIMESTAMP TRAIN PIÙ ANOMALI ===")
print(regional_train.nlargest(20, "n_anomaly").reset_index().to_string(index=False))


Caricamento CSV MTGFlow...
Score test caricati: 9,997,449
Score training caricati: 30,223,296

=== AUDIT COMPLETO ===
{
  "test": {
    "name": "test_2019",
    "rows": 9997449,
    "n_locations": 1149,
    "n_timestamps": 8701,
    "timestamp_start": "2019-01-03T11:10:00",
    "timestamp_end": "2019-12-31T23:10:00",
    "years": [
      2019
    ],
    "methods": [
      "mtgflow"
    ],
    "null_counts": {
      "location": 0,
      "timestamp": 0,
      "method": 0,
      "anomaly_score": 0,
      "threshold": 0,
      "is_anomaly": 0
    },
    "finite_scores": true,
    "finite_thresholds": true,
    "decision_mismatches": 0,
    "node_anomalies": 613164,
    "node_anomaly_rate": 0.06133204580488483,
    "windows_per_location_min": 8701,
    "windows_per_location_max": 8701,
    "nodes_per_timestamp_min": 1149,
    "nodes_per_timestamp_max": 1149,
    "threshold_variants_per_location_max": 1,
    "required_nodes_for_rare_event": 12,
    "regional_rare_timestamps": 5653,
    "regi

In [3]:
print("fraction | train rare | test rare | test timestamps")

for fraction in [0.01, 0.02, 0.05, 0.10, 0.20]:
      train_mask = regional_train["anomaly_fraction"] >= fraction
      test_mask = regional_test["anomaly_fraction"] >= fraction

      print(
          f"{fraction:>8.0%} | "
          f"{train_mask.mean():>10.2%} | "
          f"{test_mask.mean():>9.2%} | "
          f"{int(test_mask.sum()):>15,}"
      )

fraction | train rare | test rare | test timestamps
      1% |     65.65% |    64.97% |           5,653
      2% |     52.35% |    50.33% |           4,379
      5% |     31.78% |    31.56% |           2,746
     10% |     17.42% |    18.04% |           1,570
     20% |      7.13% |     7.92% |             689


## Attribuzione del massimo picco regionale

Questa diagnostica individua automaticamente il timestamp con la massima frazione di località anomale, verifica quale entità MTGFlow (`solar_irradiance_poa`, `temperature_2m` o `wind_speed_10m`) contribuisce maggiormente e confronta i valori fisici della finestra di 60 ore con la stessa finestra negli anni 2005–2018. La cella legge soltanto i dati e non crea file.

In [4]:
import xarray as xr


ENTITY_CSV = SEED_DIR / "entity_anomaly_scores.csv"
PVGIS_DIR = Path(
    "/data/SentinelPV/pvgis_data/data/pvgis_summed_irradiance"
)

if not ENTITY_CSV.is_file():
    raise FileNotFoundError(ENTITY_CSV)
if not PVGIS_DIR.is_dir():
    raise NotADirectoryError(PVGIS_DIR)

# Ogni score MTGFlow descrive l'intera finestra e usa come timestamp il suo estremo finale.
peak = pd.Timestamp(regional_test["anomaly_fraction"].idxmax())
peak_row = regional_test.loc[peak]
window_start = peak - pd.Timedelta(hours=59)

print("=== PICCO REGIONALE ===")
print(f"Timestamp finale finestra: {peak}")
print(f"Inizio finestra MTGFlow:   {window_start}")
print(f"Località anomale:          {int(peak_row['n_anomaly']):,}")
print(f"Frazione regionale:        {peak_row['anomaly_fraction']:.2%}")

# Conserva 72 ore prima e dopo il massimo per mostrare la persistenza dei tre canali.
analysis_start = peak - pd.Timedelta(hours=72)
analysis_end = peak + pd.Timedelta(hours=72)
entity_parts = []
entity_columns = [
    "location",
    "timestamp",
    "entity",
    "anomaly_score",
    "threshold",
    "is_anomaly",
]

print("\nCaricamento score per entità intorno al picco...")
for chunk in pd.read_csv(
    ENTITY_CSV,
    usecols=entity_columns,
    chunksize=1_000_000,
):
    chunk["timestamp"] = pd.to_datetime(chunk["timestamp"])
    selected = chunk[
        chunk["timestamp"].between(analysis_start, analysis_end)
    ].copy()
    if not selected.empty:
        entity_parts.append(selected)

if not entity_parts:
    raise ValueError("Nessuno score per entità trovato intorno al picco.")

entity_scores = pd.concat(entity_parts, ignore_index=True)
entity_scores["location"] = entity_scores["location"].astype(str)
if not pd.api.types.is_bool_dtype(entity_scores["is_anomaly"]):
    entity_scores["is_anomaly"] = (
        entity_scores["is_anomaly"]
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(["true", "1"])
    )
entity_scores["excess_over_threshold"] = (
    entity_scores["anomaly_score"] - entity_scores["threshold"]
)

global_peak = scores.loc[
    scores["timestamp"] == peak,
    ["location", "anomaly_score", "is_anomaly"],
].copy()
global_peak["location"] = global_peak["location"].astype(str)
global_peak = global_peak.rename(
    columns={
        "anomaly_score": "global_anomaly_score",
        "is_anomaly": "global_is_anomaly",
    }
)

entity_peak = entity_scores.loc[
    entity_scores["timestamp"] == peak
].merge(global_peak, on="location", how="left")
if entity_peak["global_is_anomaly"].isna().any():
    raise ValueError("Impossibile allineare score globali e score per entità.")

# La somma dei tre contributi deve coincidere con lo score globale canonico.
reconstructed = (
    entity_peak.groupby("location", as_index=False)["anomaly_score"]
    .sum()
    .rename(columns={"anomaly_score": "reconstructed_global_score"})
    .merge(
        global_peak[["location", "global_anomaly_score"]],
        on="location",
    )
)
max_reconstruction_error = np.abs(
    reconstructed["reconstructed_global_score"]
    - reconstructed["global_anomaly_score"]
).max()
print(
    "\nErrore massimo ricostruzione score globale:",
    f"{max_reconstruction_error:.6g}",
)


def channel_summary(frame, label):
    result = (
        frame.groupby("entity")
        .agg(
            n_locations=("location", "nunique"),
            mean_score=("anomaly_score", "mean"),
            median_score=("anomaly_score", "median"),
            mean_excess=("excess_over_threshold", "mean"),
            median_excess=("excess_over_threshold", "median"),
            entity_anomaly_rate=("is_anomaly", "mean"),
        )
        .sort_values("mean_excess", ascending=False)
    )
    print(f"\n=== {label} ===")
    print(result.to_string(float_format=lambda value: f"{value:.6f}"))
    return result


summary_all = channel_summary(entity_peak, "TUTTE LE LOCALITÀ AL PICCO")
summary_anomalous = channel_summary(
    entity_peak.loc[entity_peak["global_is_anomaly"]],
    "SOLE LOCALITÀ GLOBALMENTE ANOMALE",
)

channel_curve = (
    entity_scores.groupby(["timestamp", "entity"])["is_anomaly"]
    .mean()
    .unstack("entity")
)
print("\n=== TOP 5 TIMESTAMP PER CIASCUN CANALE ===")
for entity in channel_curve.columns:
    print(f"\n{entity}:")
    print(
        channel_curve[entity]
        .nlargest(5)
        .rename("entity_anomaly_rate")
        .to_string(float_format=lambda value: f"{value:.2%}")
    )

# Confronto fisico per località: stessa finestra calendario in ogni anno storico.
VARIABLES = (
    "solar_irradiance_poa",
    "temperature_2m",
    "wind_speed_10m",
)


def effective_poa(dataset):
    poa = dataset["solar_irradiance_poa"]
    if float(poa.max().item()) <= 0.0:
        poa = (
            dataset["direct_irradiance_tilted"]
            + dataset["diffuse_irradiance_tilted"]
        )
    return poa


def site_window_means(year):
    path = PVGIS_DIR / f"piedmont_pvgis_{year}.nc"
    if not path.is_file():
        raise FileNotFoundError(path)
    start = window_start.replace(year=year)
    end = peak.replace(year=year)
    result = {}
    with xr.open_dataset(path) as dataset:
        for variable in VARIABLES:
            data = effective_poa(dataset) if variable == "solar_irradiance_poa" else dataset[variable]
            selected = data.sel(time=slice(start, end)).transpose("location", "time")
            values = np.asarray(selected.values, dtype=np.float64)
            if values.shape[1] != 60:
                raise ValueError(
                    f"{year} {variable}: attese 60 ore, trovate {values.shape[1]}"
                )
            result[variable] = np.nanmean(values, axis=1)
    return result


print("\nCaricamento della stessa finestra nel 2005-2018...")
current = site_window_means(2019)
historical = {variable: [] for variable in VARIABLES}
for year in range(2005, 2019):
    print(f"  anno {year}")
    yearly = site_window_means(year)
    for variable in VARIABLES:
        historical[variable].append(yearly[variable])

print("\n=== CONFRONTO FISICO CON 2005-2018 ===")
for variable in VARIABLES:
    history = np.stack(historical[variable], axis=0)
    current_values = current[variable]
    historical_location_mean = np.nanmean(history, axis=0)
    historical_location_std = np.nanstd(history, axis=0, ddof=1)
    valid = (
        np.isfinite(current_values)
        & np.isfinite(historical_location_mean)
        & np.isfinite(historical_location_std)
        & (historical_location_std > 1e-6)
    )
    z_score = (
        current_values[valid] - historical_location_mean[valid]
    ) / historical_location_std[valid]
    print(f"\n{variable}")
    print(f"  media finestra 2019:        {np.nanmean(current_values):.3f}")
    print(
        "  media storica stessa data: "
        f"{np.nanmean(historical_location_mean):.3f}"
    )
    print(
        "  differenza:                 "
        f"{np.nanmean(current_values) - np.nanmean(historical_location_mean):.3f}"
    )
    print(f"  z-score mediano località:   {np.nanmedian(z_score):.3f}")
    print(f"  località con z > +2:        {np.mean(z_score > 2):.2%}")
    print(f"  località con z < -2:        {np.mean(z_score < -2):.2%}")


=== PICCO REGIONALE ===
Timestamp finale finestra: 2019-06-28 19:10:00
Inizio finestra MTGFlow:   2019-06-26 08:10:00
Località anomale:          903
Frazione regionale:        78.59%

Caricamento score per entità intorno al picco...

Errore massimo ricostruzione score globale: 0.0001087

=== TUTTE LE LOCALITÀ AL PICCO ===
                      n_locations  mean_score  median_score  mean_excess  median_excess  entity_anomaly_rate
entity                                                                                                      
temperature_2m               1149   97.319071     45.426720    88.176157      44.730572             0.788512
solar_irradiance_poa         1149   89.933129     16.986740    70.191406       0.514929             0.503916
wind_speed_10m               1149   -0.610263    -11.929664   -10.546228     -12.075909             0.159269

=== SOLE LOCALITÀ GLOBALMENTE ANOMALE ===
                      n_locations  mean_score  median_score  mean_excess  median_excess 

## Impatto del filtro normal-only sulle finestre SDE-Net

Stima quante finestre grafo-temporali di 24 ore più il target sopravvivono quando un timestamp è considerato raro se presenta almeno un nodo anomalo, almeno l'1% dei nodi anomali oppure almeno il 20%. Il calcolo viene separato per anno, quindi nessuna finestra attraversa il confine tra due anni.

In [5]:
SEQ_LEN = 24
HORIZON = 1
REQUIRED_CLEAN_HOURS = SEQ_LEN + HORIZON


def surviving_windows(name, rare_rule):
    total = 0
    kept = 0
    rare_timestamps = 0
    frame = regional_train.sort_index()

    for year, yearly in frame.groupby(frame.index.year):
        rare = rare_rule(yearly).astype(bool)
        clean = (~rare).astype(np.int8)
        valid = (
            clean.rolling(
                REQUIRED_CLEAN_HOURS,
                min_periods=REQUIRED_CLEAN_HOURS,
            ).sum()
            == REQUIRED_CLEAN_HOURS
        )
        total += max(0, len(yearly) - REQUIRED_CLEAN_HOURS + 1)
        kept += int(valid.sum())
        rare_timestamps += int(rare.sum())

    removed = total - kept
    result = {
        "rule": name,
        "rare_timestamps": rare_timestamps,
        "candidate_windows": total,
        "kept_windows": kept,
        "removed_windows": removed,
        "kept_rate": kept / total if total else float("nan"),
        "removed_rate": removed / total if total else float("nan"),
    }
    print(
        f"{name:>20}: "
        f"rare_hours={rare_timestamps:>6,}  "
        f"kept={kept:>6,}  removed={removed:>6,}  "
        f"removed_rate={result['removed_rate']:.2%}"
    )
    return result


print(
    f"Required consecutive detector-normal timestamps per SDE sample: "
    f"{REQUIRED_CLEAN_HOURS}\n"
)
normal_only_retention = pd.DataFrame(
    [
        surviving_windows(
            "any node",
            lambda frame: frame["n_anomaly"] > 0,
        ),
        surviving_windows(
            "at least 1%",
            lambda frame: frame["anomaly_fraction"] >= 0.01,
        ),
        surviving_windows(
            "at least 20%",
            lambda frame: frame["anomaly_fraction"] >= 0.20,
        ),
    ]
)

print("\n=== DETTAGLIO FILTRO NORMAL-ONLY ===")
display(normal_only_retention)


Required consecutive detector-normal timestamps per SDE sample: 25

            any node: rare_hours=24,797  kept=    21  removed=26,211  removed_rate=99.92%
         at least 1%: rare_hours=17,268  kept= 2,931  removed=23,301  removed_rate=88.83%
        at least 20%: rare_hours= 1,875  kept=22,284  removed= 3,948  removed_rate=15.05%

=== DETTAGLIO FILTRO NORMAL-ONLY ===


,rule,rare_timestamps,candidate_windows,kept_windows,removed_windows,kept_rate,removed_rate
0,any node,24797,26232,21,26211,0.000801,0.999199
1,at least 1%,17268,26232,2931,23301,0.111734,0.888266
2,at least 20%,1875,26232,22284,3948,0.849497,0.150503


## Protocollo regionale stagionale P97.5

Calcola un'unica severità regionale per timestamp come frazione di nodi MTGFlow anomali. Le quattro soglie stagionali P97.5 sono stimate esclusivamente sul 2016–2018 e applicate senza ricalibrazione al 2019. La label finale rimane binaria (`normal`/`rare`), senza categorie per temperatura, POA o vento.

In [6]:
REGIONAL_QUANTILE = 0.975
P975_SDE_SEQ_LEN = 24
P975_SDE_HORIZON = 1
P975_REQUIRED_CLEAN_HOURS = P975_SDE_SEQ_LEN + P975_SDE_HORIZON
SEASON_ORDER = ["DJF", "MAM", "JJA", "SON"]
MONTH_TO_SEASON = {
    12: "DJF", 1: "DJF", 2: "DJF",
    3: "MAM", 4: "MAM", 5: "MAM",
    6: "JJA", 7: "JJA", 8: "JJA",
    9: "SON", 10: "SON", 11: "SON",
}


def attach_season(frame):
    result = frame.copy()
    result["season"] = result.index.month.map(MONTH_TO_SEASON)
    if result["season"].isna().any():
        raise ValueError("Impossibile assegnare la stagione a tutti i timestamp.")
    return result


seasonal_train = attach_season(regional_train.sort_index())
seasonal_test = attach_season(regional_test.sort_index())

# Fit esclusivamente sul training 2016-2018.
seasonal_thresholds = (
    seasonal_train.groupby("season", observed=True)["anomaly_fraction"]
    .quantile(REGIONAL_QUANTILE)
    .reindex(SEASON_ORDER)
)
if seasonal_thresholds.isna().any():
    raise ValueError("Manca almeno una soglia stagionale P97.5.")


def apply_seasonal_thresholds(frame, thresholds):
    result = frame.copy()
    result["regional_threshold"] = result["season"].map(thresholds)
    result["is_rare_p975"] = (
        result["anomaly_fraction"] >= result["regional_threshold"]
    )
    result["event_group_p975"] = np.where(
        result["is_rare_p975"], "rare", "normal"
    )
    return result


seasonal_train = apply_seasonal_thresholds(
    seasonal_train, seasonal_thresholds
)
seasonal_test = apply_seasonal_thresholds(
    seasonal_test, seasonal_thresholds
)

season_rows = []
for season in SEASON_ORDER:
    train_part = seasonal_train.loc[seasonal_train["season"] == season]
    test_part = seasonal_test.loc[seasonal_test["season"] == season]
    season_rows.append(
        {
            "season": season,
            "p975_threshold": float(seasonal_thresholds.loc[season]),
            "required_nodes": int(
                math.ceil(seasonal_thresholds.loc[season] * 1149)
            ),
            "train_timestamps": int(len(train_part)),
            "train_rare": int(train_part["is_rare_p975"].sum()),
            "train_rare_rate": float(train_part["is_rare_p975"].mean()),
            "test_timestamps": int(len(test_part)),
            "test_rare": int(test_part["is_rare_p975"].sum()),
            "test_rare_rate": float(test_part["is_rare_p975"].mean()),
        }
    )

seasonal_p975_report = pd.DataFrame(season_rows)
print("=== SOGLIE REGIONALI STAGIONALI P97.5 ===")
display(seasonal_p975_report)

print("\n=== RIEPILOGO P97.5 ===")
print(
    f"Train rare: {int(seasonal_train['is_rare_p975'].sum()):,}/"
    f"{len(seasonal_train):,} "
    f"({seasonal_train['is_rare_p975'].mean():.2%})"
)
print(
    f"Test rare:  {int(seasonal_test['is_rare_p975'].sum()):,}/"
    f"{len(seasonal_test):,} "
    f"({seasonal_test['is_rare_p975'].mean():.2%})"
)

# Stima l'impatto del filtro normal-only su storia SDE (24h) e target (+1h).
total_windows = 0
kept_windows = 0
for year, yearly in seasonal_train.groupby(seasonal_train.index.year):
    clean = (~yearly["is_rare_p975"]).astype(np.int8)
    valid = (
        clean.rolling(
            P975_REQUIRED_CLEAN_HOURS,
            min_periods=P975_REQUIRED_CLEAN_HOURS,
        ).sum()
        == P975_REQUIRED_CLEAN_HOURS
    )
    total_windows += max(0, len(yearly) - P975_REQUIRED_CLEAN_HOURS + 1)
    kept_windows += int(valid.sum())

removed_windows = total_windows - kept_windows
seasonal_p975_retention = pd.DataFrame(
    [
        {
            "protocol": "seasonal_p975",
            "required_clean_hours": P975_REQUIRED_CLEAN_HOURS,
            "candidate_windows": total_windows,
            "kept_windows": kept_windows,
            "removed_windows": removed_windows,
            "kept_rate": kept_windows / total_windows,
            "removed_rate": removed_windows / total_windows,
        }
    ]
)
print("\n=== IMPATTO P97.5 SUL TRAINING SDE-NET ===")
display(seasonal_p975_retention)

print("\n=== 20 TIMESTAMP TEST P97.5 PIÙ SEVERI ===")
display(
    seasonal_test.loc[seasonal_test["is_rare_p975"]]
    .nlargest(20, "anomaly_fraction")
    [[
        "season",
        "n_anomaly",
        "n_scored",
        "anomaly_fraction",
        "regional_threshold",
        "event_group_p975",
    ]]
)


=== SOGLIE REGIONALI STAGIONALI P97.5 ===


,season,p975_threshold,required_nodes,train_timestamps,train_rare,train_rare_rate,test_timestamps,test_rare,test_rare_rate
0,DJF,0.233246,268,6504,164,0.025215,2101,0,0.000000
1,MAM,0.338555,389,6624,167,0.025211,2208,137,0.062047
2,JJA,0.353220,406,6624,166,0.025060,2208,162,0.073370
3,SON,0.289817,333,6552,165,0.025183,2184,3,0.001374



=== RIEPILOGO P97.5 ===
Train rare: 662/26,304 (2.52%)
Test rare:  302/8,701 (3.47%)

=== IMPATTO P97.5 SUL TRAINING SDE-NET ===


,protocol,required_clean_hours,candidate_windows,kept_windows,removed_windows,kept_rate,removed_rate
0,seasonal_p975,25,26232,24628,1604,0.938853,0.061147



=== 20 TIMESTAMP TEST P97.5 PIÙ SEVERI ===


,season,n_anomaly,n_scored,anomaly_fraction,regional_threshold,event_group_p975
timestamp,,,,,,
2019-06-28 19:10:00,JJA,903,1149,0.785901,0.35322,rare
2019-06-28 20:10:00,JJA,893,1149,0.777198,0.35322,rare
2019-06-28 18:10:00,JJA,887,1149,0.771976,0.35322,rare
2019-06-28 17:10:00,JJA,878,1149,0.764143,0.35322,rare
2019-06-28 21:10:00,JJA,868,1149,0.755440,0.35322,rare
2019-06-28 16:10:00,JJA,863,1149,0.751088,0.35322,rare
2019-06-29 18:10:00,JJA,860,1149,0.748477,0.35322,rare
2019-06-28 22:10:00,JJA,856,1149,0.744996,0.35322,rare
2019-06-29 19:10:00,JJA,852,1149,0.741514,0.35322,rare


In [7]:
dust_event = seasonal_test.loc[
      "2019-04-21":"2019-04-27",
      [
          "season",
          "n_anomaly",
          "anomaly_fraction",
          "regional_threshold",
          "is_rare_p975",
          "event_group_p975",
      ],
  ]

display(dust_event)

display(
      dust_event.assign(day=dust_event.index.normalize())
      .groupby("day")
      .agg(
          timestamps=("is_rare_p975", "size"),
          rare_hours=("is_rare_p975", "sum"),
          max_anomalous_fraction=("anomaly_fraction", "max"),
          mean_anomalous_fraction=("anomaly_fraction", "mean"),
      )
  )

,season,n_anomaly,anomaly_fraction,regional_threshold,is_rare_p975,event_group_p975
timestamp,,,,,,
2019-04-21 00:10:00,MAM,116,0.100957,0.338555,False,normal
2019-04-21 01:10:00,MAM,139,0.120975,0.338555,False,normal
2019-04-21 02:10:00,MAM,158,0.137511,0.338555,False,normal
2019-04-21 03:10:00,MAM,124,0.107920,0.338555,False,normal
2019-04-21 04:10:00,MAM,122,0.106179,0.338555,False,normal
...,...,...,...,...,...,...
2019-04-27 19:10:00,MAM,74,0.064404,0.338555,False,normal
2019-04-27 20:10:00,MAM,43,0.037424,0.338555,False,normal
2019-04-27 21:10:00,MAM,49,0.042646,0.338555,False,normal


,timestamps,rare_hours,max_anomalous_fraction,mean_anomalous_fraction
day,,,,
2019-04-21,24,0,0.149695,0.121845
2019-04-22,24,1,0.341166,0.175624
2019-04-23,24,16,0.414273,0.352082
2019-04-24,24,24,0.695387,0.594104
2019-04-25,24,24,0.609225,0.457282
2019-04-26,24,15,0.430809,0.357303
2019-04-27,24,0,0.208007,0.071004
